# ST554 Homework 10 - Streaming Data: Siona Benjamin 
In this homework assignment, we will use spark structured streaming to handle streaming data. This let's handle data as it is generated.
## Part 1: Creating Streaming data using `rate`
First we'll create streaming data using the `rate` format. This is Spark's built-in testing source that automatically generates rows at a specified rate. In this format, no external data is needed. We can use this method to practice reading, modifying, and writing our streamed data.   

In [27]:
#import necessary functions
from time import sleep
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pow

#create spark session
spark = SparkSession.builder.getOrCreate()

Now that we have our functions imported and have created our spark session, we can start streaming our data. First, we read in the rate data using `readStream`. Then, we define the modifications we want to do to the data that we're streaming in. In this case, we want to add two columns to our data, one that shows the squared value and one that shows the remainder of the value after dividing by 4 (mod 4). After defining the modifications, we write the data to memory giving it the name "squared_mod". Including `sleep(30)` lets the stream run for 30 seconds before it is stopped. 

In [28]:
#read in the rate data as a spark dataframe
rateDF = spark.readStream.format("rate").load()
#define modified dataframe with desired functions 
manipDF = rateDF.withColumn("squared_value", pow(col("value"), 2)).withColumn("mod_4", col("value")%4)
#write the stream for the modified dataframe into memory
writeDF = manipDF.writeStream.format("memory").queryName("squared_mod").start()
#sleep for 30 seconds to allow data to write 
sleep(30)
#stop writing data to memory 
writeDF.stop()

26/04/21 11:18:47 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-fdb67b8e-7ca2-4d7f-bc65-e79f5ffb857f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 11:18:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/21 11:19:17 WARN DAGScheduler: Failed to cancel job group 9fc46f26-006e-4571-baba-3478ba77bd69. Cannot find active jobs for it.
26/04/21 11:19:17 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 30, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@5df48d70] is aborting.
26/04/21 11:19:17 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 30, writer: org.apache.spark.sql.exe

After streaming our data, we can now run an sql query to return our modified dataframe and see that we now have a column with the squared values and the mod 4 values. We also see that in 30 seconds we streamed in 29 rows of data.

In [32]:
#spark sql command to show modified data
spark.sql("select * from squared_mod").show()  
print("Total number of rows:")
print(spark.sql("select * from squared_mod").count())

+--------------------+-----+-------------+-----+
|           timestamp|value|squared_value|mod_4|
+--------------------+-----+-------------+-----+
|2026-04-21 11:18:...|    0|          0.0|    0|
|2026-04-21 11:18:...|    1|          1.0|    1|
|2026-04-21 11:18:...|    2|          4.0|    2|
|2026-04-21 11:18:...|    3|          9.0|    3|
|2026-04-21 11:18:...|    4|         16.0|    0|
|2026-04-21 11:18:...|    5|         25.0|    1|
|2026-04-21 11:18:...|    6|         36.0|    2|
|2026-04-21 11:18:...|    7|         49.0|    3|
|2026-04-21 11:18:...|    8|         64.0|    0|
|2026-04-21 11:18:...|    9|         81.0|    1|
|2026-04-21 11:18:...|   10|        100.0|    2|
|2026-04-21 11:18:...|   11|        121.0|    3|
|2026-04-21 11:18:...|   12|        144.0|    0|
|2026-04-21 11:19:...|   13|        169.0|    1|
|2026-04-21 11:19:...|   14|        196.0|    2|
|2026-04-21 11:19:...|   15|        225.0|    3|
|2026-04-21 11:19:...|   16|        256.0|    0|
|2026-04-21 11:19:..

## Part 2: Using data from a CSV with a Pipeline
Now that we've used automatically generated data as our streaming source, we can move on to streaming data in from a csv source. Like before, we'll start by importing our necessary functions and creating a spark session. 

In [33]:
#import necessary functions
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pow, try_mod
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.sql.types import StructType

#create spark session
spark = SparkSession.builder.getOrCreate()

We have a csv file named "bikeDetails_for_fit.csv". We'll import this as a pandas df before converting it to a spark dataframe. We can see that this dataframe contains information for the selling conditions of different bikes, including how many miles the bike was driven for, how many owners it had, its selling price and more.

In [34]:
#read in pandas dataframe of bike data
bike_pd_df = pd.read_csv("bikeDetails_for_fit.csv")
#create spark dataframe from pandas df
bike_df = spark.createDataFrame(bike_pd_df)
bike_df.show()

+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|              NaN|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|              NaN|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|         148114.0|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|          89643.0|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|              NaN|
|    Honda CB Twister|        18000|2010| Individual|1st owner|    60000|          53857.0|
|Honda CB Hornet 160R|        78500|2018| Individual|1st owner|    17000|          87719.0|
|Royal Enfield Bul...|       180000|2008| Individual|2nd owner|    39000|       

Before actually streaming our data, we'll set up the transformations we want to apply to our dataframe. First, we'll use a SQL transformation to apply log functions to the selling price and set that as the label, apply a log function to the km_driven column, and create a dummy variable for the owner column, and only have these three columns plus the year column returned. 

In [35]:
#set up SQL transformation
sqlTrans = SQLTransformer(
    statement = """
                SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
                    CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
                    FROM __THIS__
                """
)

Next, we will use a VectorAssembler transformation to set the columns of year, log_km_driven, and one_owner as the features column. 

In [36]:
#set up vector assembler transformation
assembler = VectorAssembler(inputCols = ["year", "log_km_driven", "one_owner"], outputCol = "features", handleInvalid = 'keep')

Now that we have our transformations set up, we can set those up in a pipeline and fit that to our original dataset to easily process incoming data. 

In [37]:
#set up pipeline and fit pipeline to bike data
pipeline = Pipeline(stages = [sqlTrans, assembler])
df_pipeline = pipeline.fit(bike_df)

As a last step before we read and write in data, we'll extract the schema from our bike dataframe to format incoming data.

In [38]:
#get schema from bike dataframe
myschema = bike_df.schema

Now we are ready to set up our stream. First we'll use `readStream` with our schema, set the format to csv, and tell it to look at a folder called bike_data. Then we will write the data to the console after applying the transformations outlined in our pipeline. As we add new csv files to the bike_data folder, we can see new records being added to our wiriteDF dataframe. 

In [41]:
df = spark.readStream.schema(myschema).format("csv").option("header","true").load("bike_data")
writeDF = df_pipeline.transform(df).writeStream.outputMode("append").format("console").start()

26/04/21 11:25:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-34e69476-5a0f-4f64-be9f-0c47b2a9a534. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 11:25:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [42]:
#stop stream
writeDF.stop()

26/04/21 11:26:28 WARN DAGScheduler: Failed to cancel job group b8b82d8c-6824-4f56-9d8c-d490c6f252d5. Cannot find active jobs for it.
26/04/21 11:26:28 WARN DAGScheduler: Failed to cancel job group b8b82d8c-6824-4f56-9d8c-d490c6f252d5. Cannot find active jobs for it.


## Conclusion
In this homework, we saw how to use Spark Structured Streaming to read in, modify, and write incoming data. Using these methods allows us to easily handle newley generated data, and as we saw in Part 2, convert it to dataframes that are ready to use for machine learning purposes.